# AI Observability Demo — Data Generation Spec

> **Catalog:** `msahil.ai_observability` | **Time Range:** 30 days | **Run volume:** ~50K | **Platforms:** Databricks + New Relic + ServiceNow

## Quick Reference

| Table | Platform | Rows | Primary Key | Links To |
| --- | --- | --- | --- | --- |
| `agent_registry` | Databricks | 28 | `agent_id` | — (master) |
| `mlflow_experiments` | Databricks | 28 | `experiment_id` | agent_registry (1:1) |
| `mlflow_runs` | Databricks | ~50K | `run_id` | experiments, agent_registry via `params["agent_id"]` |
| `mlflow_run_metrics` | Databricks | ~270K | `record_id` | mlflow_runs via `run_id` |
| `ai_gateway_usage` | Databricks | ~50K | `request_id` | mlflow_runs via `request_tags["run_id"]` |
| `tool_access_logs` | Databricks | ~83K | `access_id` | mlflow_runs via `trace_id`, agent_registry via `agent_id` |
| `mcp_catalog` | Databricks | 15 | `tool_id` | tool_access_logs via `resource_path` |
| `user_profiles` | Databricks | ~133 | `user_id` | agent_registry via `agents_used` |
| `newrelic_infra_metrics` | New Relic | ~1.3M | `metric_id` | Correlated by timestamp + entity_name |
| `newrelic_apm_transactions` | New Relic | ~50K | `transaction_id` | ai_gateway_usage via `linked_gateway_request_id` |
| `servicenow_incidents` | ServiceNow | 10 | `incident_id` | agent_registry via `affected_agents`, anomaly timestamps |
| `servicenow_change_requests` | ServiceNow | 6 | `change_id` | servicenow_incidents via `parent_incident_id` |

### Cross-Platform Correlation Chain
```
mlflow_runs.run_id → ai_gateway_usage.request_tags["run_id"]
ai_gateway_usage.request_id → newrelic_apm_transactions.linked_gateway_request_id
timestamp alignment → newrelic_infra_metrics (same anomaly windows)
anomaly detection → servicenow_incidents.opened_at (minutes after anomaly start)
```

### Demo Discovery Queries
```sql
-- Find all anomaly-related failures in a time window
SELECT * FROM msahil.ai_observability.mlflow_runs
WHERE status != 'FINISHED' AND params['error_type'] != ''
ORDER BY start_time;

-- Cross-platform correlation: Gateway → APM
SELECT g.endpoint_name, g.status_code, a.external_call_duration_ms
FROM msahil.ai_observability.ai_gateway_usage g
JOIN msahil.ai_observability.newrelic_apm_transactions a
  ON g.request_id = a.linked_gateway_request_id
WHERE g.status_code >= 400;

-- Incident blast radius
SELECT number, priority, affected_users_count, mttr_minutes,
       size(affected_agents) as agents_impacted
FROM msahil.ai_observability.servicenow_incidents
ORDER BY opened_at;
```

---

## Business Context

**Customer:** E.ON SE (European energy utility)  
**Architecture:** Hub-and-Spoke model  
- **Hub:** Central IT / AI Platform team based in Germany — owns governance, platform standards, and observability infrastructure  
- **Spokes:** Regional business units operating semi-autonomously in **Germany, Italy, Sweden, Hungary, Romania, and the Netherlands**

**Demo Objective:** Demonstrate end-to-end AI Observability — monitoring, auditing, and governing AI agents deployed at scale across a multi-region, multi-cloud enterprise.

---

## Data Generation Requirements

### 1. Agent Registry
Generate a catalog of AI agents deployed across the organization:

| Field | Description |
| --- | --- |
| `agent_id` | Unique identifier (UUID) |
| `agent_name` | Descriptive name (e.g., "NL-Energy-Forecast-Agent", "DE-Customer-Support-Agent") |
| `region` | Spoke region: DE, IT, SE, HU, RO, NL |
| `cloud_provider` | AWS (Netherlands) or Azure (all others) |
| `agent_framework` | One of: `AgentBricks`, `Azure AI Agent Service`, `Amazon Bedrock Agents` |
| `deployment_env` | `production` |
| `owner_team` | Team name within the business unit |
| `created_at` | Timestamp |
| `status` | `active`, `inactive`, `deprecated` |

**Distribution:** ~60% AgentBricks (Databricks-native), ~25% Azure AI Agent Service, ~15% Amazon Bedrock Agents (across AWS regions)

---

### 2. Agent Execution Traces (based on `system.mlflow`)
Generate data modeled after `system.mlflow.runs_latest` and `system.mlflow.run_metrics_history`:

**Table: `mlflow_runs`** (mirrors `system.mlflow.runs_latest`)

| Field | Description |
| --- | --- |
| `account_id` | Account identifier |
| `workspace_id` | Workspace identifier |
| `run_id` | Unique MLflow run ID |
| `experiment_id` | MLflow experiment ID (one per agent) |
| `created_by` | User or service principal that triggered the run |
| `start_time` | Run start timestamp (last 30 days) |
| `end_time` | Run end timestamp |
| `run_name` | Descriptive run name |
| `status` | `FINISHED`, `FAILED`, `KILLED` |
| `params` | MAP<STRING,STRING> — includes `agent_id`, `region`, `model_name`, `framework` |
| `tags` | MAP<STRING,STRING> — includes `mlflow.runName`, `mlflow.source.type` |
| `aggregated_metrics` | MAP<STRING,DOUBLE> — summarized metrics (latency, tokens, cost) |

**Table: `mlflow_run_metrics`** (mirrors `system.mlflow.run_metrics_history`)

| Field | Description |
| --- | --- |
| `account_id` | Account identifier |
| `workspace_id` | Workspace identifier |
| `experiment_id` | FK to experiment |
| `run_id` | FK to mlflow_runs |
| `metric_name` | One of: `latency_ms`, `input_tokens`, `output_tokens`, `total_cost_usd`, `feedback_score`, `error_rate` |
| `metric_time` | Timestamp when metric was computed |
| `metric_step` | Step/iteration |
| `metric_value` | Numeric metric value |

**Table: `mlflow_experiments`** (mirrors `system.mlflow.experiments_latest`)

| Field | Description |
| --- | --- |
| `account_id` | Account identifier |
| `workspace_id` | Workspace identifier |
| `experiment_id` | Unique experiment ID |
| `name` | Experiment name (maps to agent name) |
| `create_time` | Creation timestamp |
| `update_time` | Last update timestamp |

**Volume:** ~500K run records across 30 days, skewed toward active agents.

---

### 3. Tool & Data Access Logs
Track what resources agents access (via Unity Catalog governance):

| Field | Description |
| --- | --- |
| `access_id` | Unique log entry |
| `trace_id` | FK to execution trace |
| `agent_id` | FK to agent registry |
| `resource_type` | `uc_table`, `uc_view`, `volume_file`, `lakebase_table`, `mcp_tool`, `external_api` |
| `resource_path` | Fully qualified path (e.g., `eon_hub.customer_360.profiles`) |
| `access_type` | `read`, `write`, `execute` |
| `row_count` | Rows accessed (for tables/views) |
| `bytes_transferred` | Data volume |
| `timestamp` | Access timestamp |
| `granted` | Boolean — was access permitted by UC policies |

---

### 4. Unity AI Gateway Traffic (based on `system.ai_gateway.usage`)
Generate data modeled after `system.ai_gateway.usage`:

| Field | Description |
| --- | --- |
| `account_id` | Account identifier |
| `workspace_id` | Workspace identifier |
| `request_id` | API proxy generated request identifier |
| `schema_version` | Event schema version |
| `endpoint_id` | UUID of the AI Gateway entity |
| `endpoint_name` | Name of the AI Gateway endpoint |
| `endpoint_tags` | MAP — static tags configured on the endpoint |
| `endpoint_metadata` | Endpoint-level metadata |
| `event_time` | Timestamp when request was received |
| `latency_ms` | End-to-end latency (request received to response forwarded) |
| `time_to_first_byte_ms` | Latency to first byte of proxy response |
| `destination_type` | Destination type (e.g., `model_serving`, `external`) |
| `destination_name` | Name of the destination endpoint or UC model |
| `destination_id` | ID of the destination |
| `destination_model` | Foundation model name (e.g., `gpt-4o`, `claude-sonnet-4`, `meta-llama-3.3-70b-instruct`) |
| `requester` | User email or service principal name |
| `requester_type` | Type of requester (user, service_principal) |
| `ip_address` | Client IP (null if Databricks internal) |
| `url` | Request URL received by AI Gateway |
| `user_agent` | Client user agent string |
| `api_type` | API type of the request (e.g., `chat`, `completions`, `embeddings`) |
| `request_tags` | MAP — tags provided in the request body |
| `input_tokens` | Total input tokens (null if unavailable) |
| `output_tokens` | Total output tokens (null if unavailable) |
| `total_tokens` | Sum of input + output tokens |
| `token_details` | Detailed token breakdown |
| `response_content_type` | Content-Type header (streaming vs non-streaming) |
| `status_code` | Final HTTP status code (200, 429, 500, etc.) |
| `routing_information` | Detailed routing info for primary + fallbacks |
| `invocation_id` | Unique ID per individual inference call |
| `invocation_metadata` | System-generated metadata about the call |

**Link to agents:** Use `request_tags` or `requester` to map back to `agent_id` in the agent registry.

---

### 5. MCP Catalog — Registered Tools
Tools available to agents via MCP (Model Context Protocol):

| Field | Description |
| --- | --- |
| `tool_id` | Unique identifier |
| `tool_name` | e.g., `get_customer_usage`, `query_energy_prices`, `file_search` |
| `tool_type` | `uc_function`, `vector_search`, `rest_api`, `sql_query` |
| `catalog_path` | UC path if applicable |
| `description` | Natural language description |
| `owner_region` | Region that owns/maintains the tool |
| `call_count_30d` | Usage in last 30 days |
| `avg_latency_ms` | Average execution time |

---

### 6. User Profiles
Internal users interacting with the agents:

| Field | Description |
| --- | --- |
| `user_id` | Unique identifier |
| `user_email` | Corporate email |
| `region` | Business unit region |
| `department` | e.g., Operations, Customer Service, Trading, Engineering |
| `role` | `analyst`, `engineer`, `manager`, `executive` |
| `agents_used` | Array of agent_ids this user has interacted with |

---

## Correlated Anomaly Scenarios

Generate **realistic, correlated anomalies** that appear simultaneously in both the AI Gateway and MLflow trace data. These should be detectable through cross-table analysis and demonstrate the value of unified observability.

---

### Scenario 1: Regional Model Provider Outage (Netherlands / AWS)
**Trigger:** Azure OpenAI endpoint in NL region becomes unavailable for ~2 hours  
**AI Gateway signals:**
- Spike in `status_code` = 503 for `endpoint_name` = `nl-gpt4o-endpoint`
- `fallback_triggered` = true in `routing_information`
- `latency_ms` jumps 3–5x as requests route to fallback (DE region)
- `destination_model` shifts from `gpt-4o` to `gpt-4o-mini` (degraded fallback)

**MLflow signals:**
- NL agents show `status` = `FAILED` spike (~40% failure rate during window)
- `aggregated_metrics["latency_ms"]` increases for surviving requests
- `aggregated_metrics["feedback_score"]` drops (degraded quality from fallback model)
- `params["model_name"]` shows unexpected `gpt-4o-mini` values for agents configured for `gpt-4o`

---

### Scenario 2: Token Budget Exhaustion (Italy — Customer Service Agent)
**Trigger:** IT-Customer-Support-Agent starts receiving unusually long customer complaints (regulatory season), exceeding context windows  
**AI Gateway signals:**
- `status_code` = 400 (context overflow) spikes for `endpoint_name` = `it-customer-agent-endpoint`
- `input_tokens` values exceed 120K (beyond model limit)
- `api_type` = `chat` requests show growing `input_tokens` trend over 3 days

**MLflow signals:**
- `metric_name` = `error_rate` climbs from 2% → 18% over 3 days for IT agents
- `status` = `FAILED` with `params["error_type"]` = `context_overflow`
- `metric_name` = `input_tokens` shows upward trend (avg 80K → 130K)
- Correlated drop in `feedback_score` as truncated responses frustrate users

---

### Scenario 3: Rate Limiting Cascade (Hub — Peak Hours)
**Trigger:** Monday 9 AM CET — all regions come online simultaneously, exceeding shared AI Gateway rate limits  
**AI Gateway signals:**
- `status_code` = 429 spike between 08:45–09:30 CET across multiple endpoints
- Concentrated in `requester_type` = `service_principal` (automated agents)
- `latency_ms` = null (requests rejected before processing)
- Affects DE, HU, RO agents sharing the same gateway endpoint

**MLflow signals:**
- Burst of `status` = `FAILED` runs with short `end_time - start_time` (immediate failures)
- `metric_name` = `latency_ms` shows bimodal distribution: very fast failures + slow retries
- Retry storms visible: same `params["request_id"]` appears in multiple runs
- Agents with retry logic show 3–5x normal run count during window

---

### Scenario 4: Guardrail Blocking Surge (Germany — HR Agent)
**Trigger:** DE-HR-Policy-Agent starts receiving adversarial prompts attempting to extract salary data  
**AI Gateway signals:**
- `invocation_metadata` shows `guardrail_action` = `blocked` for PII detection
- `status_code` = 200 but `output_tokens` = 0 (blocked at output)
- `request_tags` contain `guardrail_triggered: ["pii_detection", "sensitive_data"]`
- Concentrated from 3–4 specific `requester` values (compromised accounts)

**MLflow signals:**
- `status` = `FINISHED` but `aggregated_metrics["output_tokens"]` = 0
- `metric_name` = `feedback_score` drops to 1.0 (users get empty responses)
- `params["guardrail_blocks"]` count spikes from 0.1% → 12% of requests
- Tool access logs show denied `granted` = false for `eon_hub.hr.salary_bands`

---

### Scenario 5: Model Drift — Degraded Quality (Sweden — Energy Forecasting)
**Trigger:** SE-Energy-Forecast-Agent's underlying model updated by provider; predictions become less accurate over 7 days  
**AI Gateway signals:**
- No errors — all `status_code` = 200, normal `latency_ms`
- Subtle: `destination_model` version changes (e.g., `gpt-4o-2025-05-13` → `gpt-4o-2025-06-01`)
- `output_tokens` average decreases (model giving shorter, less detailed forecasts)

**MLflow signals:**
- `metric_name` = `feedback_score` gradual decline: 4.2 → 3.1 over 7 days
- `metric_name` = `total_cost_usd` decreases (shorter outputs = lower cost, but worse quality)
- Downstream metric: `forecast_accuracy_mape` degrades from 3.5% → 8.2%
- No `FAILED` runs — silent quality degradation only visible through metrics

---

### Scenario 6: Data Access Anomaly — Unauthorized Lateral Movement (Romania)
**Trigger:** RO-Operations-Agent unexpectedly starts querying cross-region tables it shouldn't access  
**AI Gateway signals:**
- Normal traffic patterns — no gateway-level anomalies
- `request_tags` show new tool calls: `query_de_customer_data`, `access_hub_financials`

**MLflow signals:**
- New `params["tools_called"]` values appear that weren't in training data
- `metric_name` = `latency_ms` increases (cross-region data fetches)
- Run duration extends as agent attempts multi-hop data access

**Tool Access Logs (§3) correlation:**
- `granted` = false spikes for `agent_id` = RO agent accessing `eon_hub.finance.*`
- `resource_path` values outside normal RO scope: `eon_de.customer_360.*`, `eon_hub.hr.*`
- Sequential access pattern suggests prompt injection (user manipulating agent behavior)

---

### Scenario 7: Agent Reasoning Loop / Runaway Cost (Hungary — Operations Agent)
**Trigger:** HU-Grid-Operations-Agent gets stuck in a tool-calling loop due to ambiguous user query — repeatedly calls the same MCP tool expecting different results  
**AI Gateway signals:**
- Single `request_id` generates 50+ `invocation_id` entries (normal is 3–5)
- `total_tokens` per request balloons to 500K+ (multi-turn tool loop)
- `latency_ms` exceeds 120,000ms (2+ minutes for a single conversation)
- `input_tokens` grows geometrically per invocation (context accumulates)

**MLflow signals:**
- `metric_name` = `total_cost_usd` spikes 20x for individual runs (single run costs $2+ vs normal $0.10)
- `end_time - start_time` exceeds 3 minutes (normal: 5–15 seconds)
- `metric_name` = `tool_calls_count` shows values of 40–60 (normal: 2–5)
- `params["tools_called"]` shows repeated identical tool: `check_grid_status` called 50+ times

**MCP Catalog (§5) correlation:**
- `call_count_30d` for `check_grid_status` tool shows 10x spike on anomaly day
- `avg_latency_ms` for the tool remains normal (proving the tool isn't broken — the agent is looping)

---

### Scenario 8: MCP Tool Degradation / Cascading Agent Failure (Cross-Region)
**Trigger:** A shared MCP tool (`query_energy_prices` — UC function backed by a Lakebase table) becomes slow due to upstream data pipeline delay, causing dependent agents across 4 regions to timeout  
**AI Gateway signals:**
- No gateway-level errors (LLM calls succeed)
- But `latency_ms` increases for all agents using the affected tool
- Multiple endpoints show correlated latency spike without any model-side issue

**MLflow signals:**
- `metric_name` = `latency_ms` spikes for DE, IT, HU, SE agents simultaneously
- `status` = `KILLED` (timeout) for agents with strict SLA enforcement
- `params["tools_called"]` contains `query_energy_prices` in all affected runs
- `metric_name` = `tool_latency_ms` for `query_energy_prices` jumps from 200ms → 15,000ms

**Tool Access Logs (§3) correlation:**
- `resource_type` = `lakebase_table` + `resource_path` = `eon_hub.energy.spot_prices`
- `bytes_transferred` drops to 0 (stale/unavailable data)
- Timestamp gap visible — no successful access for 45 minutes

**MCP Catalog (§5) correlation:**
- `avg_latency_ms` for `query_energy_prices` jumps from 180ms → 14,500ms
- Demonstrates cross-region blast radius of a single shared tool

---

### Scenario 9: Shadow AI / Unmonitored Agent Path (Netherlands)
**Trigger:** A developer in NL bypasses the AI Gateway by calling Amazon Bedrock directly (not through the governed endpoint), creating an observability blind spot  
**AI Gateway signals:**
- NL agent `nl-bedrock-dev-agent` shows ZERO gateway traffic for 6 hours (normally 200+ requests/hour)
- Gap is visible as a sudden drop-to-zero in the time series

**MLflow signals:**
- Agent continues to produce `FINISHED` runs (it's still working)
- But `params["gateway_request_id"]` is NULL (no gateway correlation)
- `aggregated_metrics["total_cost_usd"]` = 0 (costs not captured)
- Quality metrics still logged but cost/governance data missing

**Audit trail gap:**
- Tool Access Logs show continued `resource_type` = `uc_table` access from the agent
- But no corresponding AI Gateway record for the same time window
- Demonstrates compliance risk: agent is active but invisible to central governance

---

### Scenario 10: Cross-Agent Cascading Failure (Hub-to-Spoke)
**Trigger:** DE-Orchestrator-Agent (Hub) calls downstream spoke agents (IT, HU, RO) as sub-agents. When IT-agent fails, the orchestrator retries aggressively, overwhelming HU and RO agents  
**AI Gateway signals:**
- `requester` = `de-orchestrator-sp@eon.com` shows 5x normal request volume
- IT endpoint: `status_code` = 500 sustained
- HU + RO endpoints: `status_code` = 429 (rate limited by orchestrator's retry storm)
- `routing_information` shows no fallback available (all spokes saturated)

**MLflow signals:**
- DE-Orchestrator-Agent: `status` = `FAILED`, `end_time - start_time` = timeout (300s)
- IT spoke agents: `status` = `FAILED` (root cause)
- HU + RO spoke agents: `status` = `FAILED` (collateral damage from retry storm)
- `metric_name` = `error_rate` spikes to 80%+ across 3 regions simultaneously
- Causal chain visible in timestamps: IT fails first → DE retries → HU/RO overwhelmed

**User Profiles (§6) correlation:**
- `department` = `Operations` users across IT, HU, RO all report degraded experience
- Demonstrates blast radius: single spoke failure → Hub retry → multi-spoke cascade

---

### Anomaly Injection Parameters

| Scenario | Time Window | Affected Region | Severity | Detection Difficulty |
| --- | --- | --- | --- | --- |
| 1. Provider Outage | Day 12, 14:00–16:00 CET | NL | High | Easy (clear error codes) |
| 2. Token Exhaustion | Days 8–10, gradual | IT | Medium | Medium (trend-based) |
| 3. Rate Limit Cascade | Days 5, 12, 19 (Mondays 9 AM) | DE, HU, RO | High | Easy (periodic pattern) |
| 4. Guardrail Surge | Day 18, 10:00–14:00 CET | DE | Critical | Medium (needs cross-table join) |
| 5. Model Drift | Days 20–27, continuous | SE | Low | Hard (no errors, only quality) |
| 6. Lateral Movement | Day 22, 03:00–05:00 CET | RO | Critical | Hard (requires access log correlation) |
| 7. Reasoning Loop | Day 15, 11:00–12:30 CET | HU | High | Medium (cost/duration spike) |
| 8. Tool Degradation | Day 9, 06:00–06:45 CET | DE, IT, HU, SE | High | Hard (no LLM errors, tool-level) |
| 9. Shadow AI Gap | Day 14, 20:00 – Day 15, 02:00 CET | NL | Critical | Hard (absence of data) |
| 10. Cross-Agent Cascade | Day 25, 09:15–09:45 CET | DE→IT→HU→RO | Critical | Medium (temporal correlation) |

---

### Implementation Notes for Anomaly Generation
- Inject anomalies AFTER generating baseline "healthy" data
- Ensure timestamps are consistent across Gateway + MLflow + Access Logs for the same event
- Use the same `request_id` / `run_id` linkage so cross-table JOINs reveal the correlation
- Healthy baseline: 95% success rate, avg latency 200–800ms, feedback 3.8–4.5
- Each anomaly should be discoverable via at least 2 different query patterns (single-table alert + cross-table root cause)

---

## Integrated Observability Storyline: Databricks + New Relic + ServiceNow

### The Narrative: "From AI Signal → Infrastructure Correlation → Automated Remediation"

This demo tells the story of a **single incident lifecycle** across three platforms, proving that no single tool can deliver full AI observability alone:

| Platform | Role | Answers |
| --- | --- | --- |
| **Databricks** | AI-Native Observability | "What failed in our AI estate, why, and what's the governance impact?" |
| **New Relic** | Infrastructure Correlation | "Was this an AI problem or an infrastructure problem?" |
| **ServiceNow** | Automated Response & Remediation | "Who's responsible, what's the SLA impact, and what's the fix?" |

---

### Hero Story: Scenario 10 — Cross-Agent Cascading Failure

**Act 1 — Databricks Detects (T+0 min)**
- AI Gateway shows IT endpoint returning 500s
- MLflow traces reveal DE-Orchestrator retry storm → HU/RO rate limited
- Unity Catalog access logs confirm blast radius across 4 regions
- *Platform value: AI-specific root cause analysis, cross-table correlation*

**Act 2 — New Relic Correlates (T+2 min)**
- APM dashboard shows Azure OpenAI endpoint in IT region with elevated error rate
- Network traces reveal upstream connectivity degradation to `italynorth.api.cognitive.microsoft.com`
- Compute metrics show Databricks serving endpoint CPU spike (retry handling)
- *Platform value: Proves root cause is infrastructure, not agent logic*

**Act 3 — ServiceNow Remediates (T+5 min)**
- P1 incident auto-created via Databricks SQL Alert → webhook
- Ticket auto-populated: affected agents, impacted users, CMDB CI linkage
- Runbook attached: "Cross-region fallback enablement procedure"
- Change Request generated: update AI Gateway routing rules for IT region
- *Platform value: Automated triage, SLA tracking, audit trail*

---

### Supporting Demo Paths

| Scenario | Databricks Signal | New Relic Signal | ServiceNow Action |
| --- | --- | --- | --- |
| 1. Provider Outage (NL) | 503s + fallback routing | Azure status page correlation, DNS resolution spike | P2 incident, vendor escalation workflow |
| 5. Model Drift (SE) | Feedback decline, no errors | No infra signal (proves NOT infra) | Quality SLA breach ticket after 3-day threshold |
| 7. Reasoning Loop (HU) | Cost spike, 50+ tool calls | Serving endpoint memory/CPU spike | P2 budget alert, auto-kill runbook |
| 8. Tool Degradation (cross-region) | Tool latency 200ms→15s | Lakebase connection pool exhaustion | P1 data pipeline incident, blast radius assessment |
| 9. Shadow AI (NL) | Gateway traffic gap (zero) | Direct Bedrock API calls visible in VPC flow logs | Compliance violation, security review ticket |

---

### Key Demo Message

> **Databricks** owns the *AI-native signal* (traces, cost, quality, governance).  
> **New Relic** owns the *infrastructure signal* (APM, network, compute health).  
> **ServiceNow** owns the *action* (incidents, remediation, compliance).  
> The integrated stack eliminates blind spots that exist when any platform operates alone.

---

### 7. New Relic Infrastructure Metrics
Simulated APM and infrastructure telemetry correlated with AI anomaly events:

**Table: `newrelic_infra_metrics`** — Time-series infrastructure metrics

| Field | Description |
| --- | --- |
| `metric_id` | Unique metric record ID |
| `timestamp` | Metric collection timestamp (15-second intervals) |
| `region` | Region: DE, IT, SE, HU, RO, NL |
| `account_id` | FK to E.ON account (per-country) |
| `entity_type` | `AI_GATEWAY_ENDPOINT`, `MODEL_SERVING_ENDPOINT`, `AZURE_OPENAI`, `AWS_BEDROCK`, `LAKEBASE_INSTANCE`, `NETWORK_INTERFACE` |
| `entity_name` | Specific entity (e.g., `de-gpt4o-endpoint`, `it-azure-openai-instance`) |
| `metric_name` | One of: `cpu_percent`, `memory_percent`, `request_rate_per_sec`, `error_rate_percent`, `latency_p50_ms`, `latency_p99_ms`, `connection_pool_active`, `connection_pool_max`, `network_latency_ms`, `dns_resolution_ms`, `bytes_in_per_sec`, `bytes_out_per_sec`, `active_threads`, `queue_depth` |
| `metric_value` | Numeric value |
| `alert_severity` | `NORMAL`, `WARNING`, `CRITICAL`, or null |
| `alert_policy_name` | New Relic alert policy name (if threshold breached) |

**Table: `newrelic_apm_transactions`** — Application-level traces

| Field | Description |
| --- | --- |
| `transaction_id` | Unique APM transaction ID |
| `timestamp` | Transaction timestamp |
| `region` | Region |
| `service_name` | `ai-gateway-proxy`, `model-serving-{region}`, `agent-runtime-{region}` |
| `transaction_name` | e.g., `POST /serving-endpoints/{name}/invocations` |
| `duration_ms` | Transaction duration |
| `status_code` | HTTP response code |
| `error` | Boolean — did this transaction result in an error? |
| `external_call_count` | Number of external service calls |
| `external_call_duration_ms` | Time spent in external calls |
| `database_call_count` | Number of database calls |
| `database_call_duration_ms` | Time spent in database calls |
| `linked_gateway_request_id` | FK to `ai_gateway_usage.request_id` (for cross-platform correlation) |

**Anomaly Correlation in New Relic Data:**
- Scenario 1 (NL Outage): `network_latency_ms` + `dns_resolution_ms` spike for NL→Azure, `error_rate_percent` jumps
- Scenario 7 (Reasoning Loop): `cpu_percent` + `memory_percent` + `active_threads` spike on HU serving endpoint
- Scenario 8 (Tool Degradation): `connection_pool_active` approaches `connection_pool_max` on Lakebase instance
- Scenario 9 (Shadow AI): VPC flow logs show `bytes_out_per_sec` to Bedrock IP range while gateway shows zero traffic
- Scenario 10 (Cascade): `request_rate_per_sec` spikes 5x on DE orchestrator, `queue_depth` grows on HU/RO endpoints

---

### 8. ServiceNow Incidents & Change Requests
ITSM records auto-generated from observability alerts:

**Table: `servicenow_incidents`** — Incident tickets

| Field | Description |
| --- | --- |
| `incident_id` | ServiceNow sys_id (INC format) |
| `number` | Incident number (e.g., INC0012345) |
| `short_description` | Auto-generated summary from alert |
| `description` | Detailed description including affected agents, regions, metrics |
| `priority` | P1, P2, P3, P4 |
| `state` | `New`, `In Progress`, `On Hold`, `Resolved`, `Closed` |
| `category` | `AI Platform`, `Infrastructure`, `Security`, `Compliance` |
| `subcategory` | `Model Serving`, `AI Gateway`, `Data Pipeline`, `Guardrail Violation`, `Cost Anomaly`, `Shadow AI` |
| `assignment_group` | Responsible team (e.g., `AI Platform - Hub`, `SRE - NL`, `Security Operations`) |
| `assigned_to` | Individual assignee email |
| `opened_at` | Incident creation timestamp |
| `resolved_at` | Resolution timestamp (null if open) |
| `closed_at` | Closure timestamp (null if open) |
| `impact` | `1-Enterprise`, `2-Region`, `3-Team`, `4-Individual` |
| `urgency` | `1-Critical`, `2-High`, `3-Medium`, `4-Low` |
| `cmdb_ci` | Configuration Item (linked infra component) |
| `source_alert_id` | FK to Databricks SQL Alert or New Relic alert that triggered the incident |
| `source_system` | `Databricks`, `New Relic`, `Manual` |
| `affected_regions` | Array of impacted regions |
| `affected_agents` | Array of impacted agent_ids |
| `affected_users_count` | Estimated number of impacted end users |
| `mttr_minutes` | Mean time to resolve (calculated) |
| `root_cause` | Post-mortem root cause category |
| `resolution_notes` | How it was resolved |

**Table: `servicenow_change_requests`** — Change tickets triggered by incidents

| Field | Description |
| --- | --- |
| `change_id` | ServiceNow sys_id (CHG format) |
| `number` | Change number (e.g., CHG0005678) |
| `short_description` | Change summary |
| `type` | `Standard`, `Normal`, `Emergency` |
| `state` | `New`, `Assess`, `Authorize`, `Scheduled`, `Implement`, `Review`, `Closed` |
| `risk` | `Low`, `Medium`, `High` |
| `parent_incident_id` | FK to `servicenow_incidents.incident_id` |
| `requested_by` | Who requested the change |
| `assignment_group` | Implementation team |
| `planned_start` | Scheduled implementation start |
| `planned_end` | Scheduled implementation end |
| `description` | Detailed change description |
| `cmdb_ci` | Configuration Item being changed |
| `backout_plan` | Rollback procedure |

**Incident-to-Anomaly Mapping:**

| Scenario | Incident Priority | Category | Time to Detect | MTTR |
| --- | --- | --- | --- | --- |
| 1. Provider Outage | P2 | Infrastructure | 2 min (auto) | 95 min (vendor resolution) |
| 2. Token Exhaustion | P3 | AI Platform | 4 hours (trend alert) | 30 min (config update) |
| 3. Rate Limit Cascade | P2 | AI Platform | 1 min (auto) | 15 min (rate limit increase) |
| 4. Guardrail Surge | P1 | Security | 3 min (auto) | 45 min (account lockout + investigation) |
| 5. Model Drift | P3 | AI Platform | 72 hours (SLA breach) | 120 min (model pin to previous version) |
| 6. Lateral Movement | P1 | Security | 8 min (access log alert) | 20 min (agent isolation + credential rotation) |
| 7. Reasoning Loop | P2 | AI Platform | 5 min (cost alert) | 10 min (agent kill + query fix) |
| 8. Tool Degradation | P1 | Data Pipeline | 12 min (multi-region alert) | 45 min (pipeline restart) |
| 9. Shadow AI | P1 | Compliance | 6 hours (gap detection) | 180 min (audit + remediation) |
| 10. Cross-Agent Cascade | P1 | Infrastructure | 2 min (auto) | 30 min (circuit breaker + routing fix) |

---

## Technical Constraints

- **Output format:** Delta tables in Unity Catalog
- **Catalog:** `msahil`
- **Schema:** `ai_observability`
- **Libraries:** Use `dbldatagen` (Databricks Labs Data Generator) or `Faker` + PySpark
- **Referential integrity:** Maintain FK relationships across all tables
- **Time range:** Last 30 days with realistic daily/hourly patterns (business hours peak)
- **Regional realism:** Agent names, user names, and departments should reflect each country's context

---

## Demo Execution Guide

### Notebooks in this project
| Notebook | Purpose |
| --- | --- |
| `0 - Setup` (this file) | Data generation spec — the single source of truth for schema, scenarios, and storyline |
| `1 - Demo Data` | Executable notebook that generates all 12 tables into `msahil.ai_observability` |

### Running the data generation
1. Open `1 - Demo Data` and run all cells sequentially (top to bottom)
2. Total runtime: ~3–5 minutes on Serverless
3. The final cell prints a summary with row counts for all 12 tables

### Anomaly timeline (Day = offset from START_DATE)
```
Day  5 │ Mon 9 AM  │ Rate Limit Cascade (DE/HU/RO)
Day  8 │ gradual   │ Token Exhaustion begins (IT)
Day  9 │ 06:00     │ Tool Degradation + Lakebase connection pool exhaustion
Day 12 │ Mon 9 AM  │ Rate Limit Cascade (repeat)
       │ 14:00 CET │ Provider Outage (NL, 2 hours)
Day 14 │ 20:00     │ Shadow AI gap starts (NL Bedrock agent bypasses gateway)
Day 15 │ 11:00     │ Reasoning Loop (HU, 90 minutes)
Day 18 │ 10:00     │ Guardrail Surge (DE HR, 4 hours)
Day 19 │ Mon 9 AM  │ Rate Limit Cascade (repeat)
Day 20 │ gradual   │ Model Drift begins (SE, 7-day degradation)
Day 22 │ 03:00     │ Lateral Movement (RO, off-hours security event)
Day 25 │ 09:15     │ Cross-Agent Cascade (DE→IT→HU→RO, 30 minutes)
```

---

### Demo Outline — 3 Scenarios

The demo walks through **three hero scenarios** in sequence, each showcasing a different detection pattern and cross-platform correlation:

---

#### Scenario A: Cross-Agent Cascading Failure (Scenario 10)
**Theme:** "Detect → Correlate → Remediate" — the full lifecycle

| Step | Platform | What to show | Query / View |
| --- | --- | --- | --- |
| 1 | Databricks | IT agent failures trigger orchestrator retry storm | `mlflow_runs WHERE status='FAILED' AND start_time BETWEEN Day 25 09:15 AND 09:45` |
| 2 | Databricks | AI Gateway shows 429s cascading to HU/RO | `ai_gateway_usage WHERE status_code IN (429,500,504) AND event_time BETWEEN ...` |
| 3 | New Relic | Azure OpenAI error rate spike in IT (root cause = infra) | `newrelic_infra_metrics WHERE entity_name='it-azure-openai-instance' AND alert_severity='CRITICAL'` |
| 4 | New Relic | DE orchestrator queue depth + request rate spike | `newrelic_infra_metrics WHERE entity_name='de-orchestrator-serving' AND metric_name='queue_depth'` |
| 5 | ServiceNow | P1 incident auto-created, 4 regions affected | `servicenow_incidents WHERE number='INC0012010'` |
| 6 | ServiceNow | Emergency change request: circuit breaker implementation | `servicenow_change_requests WHERE number='CHG0005003'` |

**Key message:** Single spoke failure → Hub retry storm → multi-region cascade. Only cross-platform correlation reveals the root cause was Azure infra, not agent logic.

---

#### Scenario B: Token Budget Exhaustion (Scenario 2)
**Theme:** "Gradual degradation detection" — trend-based alerting vs threshold-based

| Step | Platform | What to show | Query / View |
| --- | --- | --- | --- |
| 1 | Databricks | IT-Customer-Support-Agent error_rate trending up over 3 days | `mlflow_run_metrics WHERE experiment_id=(IT agent) AND metric_name='error_rate' AND metric_time BETWEEN Day 8 AND Day 10` |
| 2 | Databricks | Input tokens growing: avg 80K → 130K (context overflow) | `mlflow_run_metrics WHERE metric_name='input_tokens' AND experiment_id=(IT agent)` |
| 3 | Databricks | AI Gateway shows 400 status codes with input_tokens > 120K | `ai_gateway_usage WHERE status_code=400 AND endpoint_name='it-customer-agent-endpoint'` |
| 4 | Databricks | Feedback score correlated drop | `mlflow_run_metrics WHERE metric_name='feedback_score' AND experiment_id=(IT agent)` |
| 5 | New Relic | No infrastructure signal — proves it's an AI-layer problem | `newrelic_infra_metrics WHERE region='IT' AND alert_severity IS NOT NULL` → empty |
| 6 | ServiceNow | P3 ticket created after 4 hours (trend alert threshold) | `servicenow_incidents WHERE number='INC0012002'` |

**Key message:** This failure is invisible to infrastructure monitoring. Only AI-native observability (MLflow token metrics + Gateway error codes) catches the gradual degradation before it becomes a full outage. The fix is a config change (chunking strategy), not an infra change.

---

#### Scenario C: Model Drift — Silent Quality Degradation (Scenario 5)
**Theme:** "No errors, no alerts, just declining quality" — the hardest problem

| Step | Platform | What to show | Query / View |
| --- | --- | --- | --- |
| 1 | Databricks | All requests succeed — 100% status=FINISHED, 200 status codes | `mlflow_runs WHERE params['agent_id']=(SE agent) AND status='FINISHED' AND start_time BETWEEN Day 20 AND 27` |
| 2 | Databricks | But feedback_score declining: 4.2 → 3.1 over 7 days | `mlflow_run_metrics WHERE metric_name='feedback_score' AND experiment_id=(SE agent) ORDER BY metric_time` |
| 3 | Databricks | Gateway shows model version change (subtle) | `ai_gateway_usage WHERE endpoint_name='se-forecast-endpoint' AND destination_model LIKE '%06-01%'` |
| 4 | Databricks | Output tokens decreasing (shorter, less detailed forecasts) | `mlflow_run_metrics WHERE metric_name='output_tokens' AND experiment_id=(SE agent)` |
| 5 | New Relic | Zero infra alerts — everything looks healthy | Infra dashboard shows all green for SE region |
| 6 | ServiceNow | P3 ticket only after 72 hours (SLA breach threshold) | `servicenow_incidents WHERE number='INC0012005'` |

**Key message:** This is the nightmare scenario for any AI platform team. Traditional monitoring sees nothing wrong. Only feedback-loop observability (user scores + output quality metrics) catches the drift. Resolution requires model version pinning — a capability only possible with AI Gateway version tracking.

---

### Demo Flow Summary

```
Scenario A (Cascade)    → Shows the FULL integrated stack working together
Scenario B (Token)      → Shows AI-native detection that infra monitoring misses
Scenario C (Drift)      → Shows silent failure that NOTHING catches except quality metrics
```

**Opening:** "Let me show you three very different failure modes — each one invisible to at least one of these platforms alone."

**Closing:** "Without unified AI observability, Scenario A looks like an agent bug, Scenario B goes undetected for days, and Scenario C goes undetected for weeks."

---

## Demo Execution Guide

### Notebooks in this project
| Notebook | Purpose |
| --- | --- |
| `0 - Setup` (this file) | Data generation spec — the single source of truth for schema, scenarios, and storyline |
| `1 - Demo Data` | Executable notebook that generates all 12 tables into `msahil.ai_observability` |

### Running the data generation
1. Open `1 - Demo Data` and run all cells sequentially (top to bottom)
2. Total runtime: ~3–5 minutes on Serverless
3. The final cell prints a summary with row counts for all 12 tables

### Anomaly timeline (Day = offset from START_DATE)
```
Day  5 │ Mon 9 AM  │ Rate Limit Cascade (DE/HU/RO)
Day  8 │ 06:00     │ Tool Degradation starts + Lakebase connection pool exhaustion
Day  9 │ gradual   │ Token Exhaustion begins (IT)
Day 12 │ Mon 9 AM  │ Rate Limit Cascade (repeat)
     │ 14:00 CET │ Provider Outage (NL, 2 hours)
Day 14 │ 20:00     │ Shadow AI gap starts (NL Bedrock agent bypasses gateway)
Day 15 │ 11:00     │ Reasoning Loop (HU, 90 minutes)
Day 18 │ 10:00     │ Guardrail Surge (DE HR, 4 hours)
Day 19 │ Mon 9 AM  │ Rate Limit Cascade (repeat)
Day 20 │ gradual   │ Model Drift begins (SE, 7-day degradation)
Day 22 │ 03:00     │ Lateral Movement (RO, off-hours security event)
Day 25 │ 09:15     │ Cross-Agent Cascade (DE→IT→HU→RO, 30 minutes)
```

### Suggested demo flow (aligned with 3 hero scenarios below)
1. **Set the stage** — show the E.ON Hub-and-Spoke map, 28 agents across 6 regions, healthy baseline (Days 1–4)
2. **Scenario A: Cross-Agent Cascade** — full-stack story showing Databricks detection → New Relic infra correlation → ServiceNow remediation
3. **Scenario B: Token Exhaustion** — gradual trend story where only AI-native monitoring catches the degradation (infra sees nothing)
4. **Scenario C: Model Drift** — silent failure story where zero errors fire, zero infra alerts trigger, only quality metrics reveal the problem
5. **Wrap-up** — side-by-side comparison: what each platform sees vs misses, and why the integrated stack is necessary

### Key talking points (per scenario)

**Scenario A (Cascade):**
- Without New Relic infra correlation, this looks like an agent logic bug in Databricks — but the root cause was Azure OpenAI connectivity in IT region
- Without Databricks, New Relic sees an API error but has no visibility into the retry storm cascading to HU/RO agents
- Without ServiceNow automation, MTTR would be hours of manual triage instead of 30 minutes with auto-generated P1 + circuit breaker change request

**Scenario B (Token Exhaustion):**
- New Relic shows **zero infrastructure signals** — all healthy. This proves the failure is at the AI layer, not infra
- Only Databricks (MLflow metrics + AI Gateway) can detect the *trend* of input_tokens growing over 3 days
- The fix is a config change (input chunking), not an infrastructure change — wrong team would be paged without AI-native observability

**Scenario C (Model Drift):**
- This is the nightmare scenario: 100% success rate, 200 status codes, zero alerts. Traditional monitoring sees nothing wrong
- Only MLflow feedback_score + output quality metrics reveal the problem — after 7 days
- AI Gateway version tracking (`destination_model` field) provides the forensic evidence: the provider silently updated the model
- Without quality-loop observability, this goes undetected until business users escalate weeks later